<a href="https://colab.research.google.com/github/jiminmini/mini/blob/ESAA_OB/9_12_%ED%95%84%EC%82%AC%EA%B3%BC%EC%A0%9C_%EC%B5%9C%EC%A2%85.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**[개념 정리]**

##**[부스팅]**

- 부스팅: 약한 학습기를 여러 개 연결하여 강한 학습기를 만드는 앙상블 방법

##**[에이다 부스트]**
- 에이다 부스트: 이전 예측기를 보완하는 새로운 예측기를 만드는 방법


##**[그레이디언트 부스팅]**
- 그레이디언트 부스팅: 앙상블에 이전까지의 오차를 보정하도록 예측기를 순차적으로 추가함

- 확률적 그레이디언트 부스팅: 각 트리가 훈련할 때 사용할 훈련 샘플의 비율을 지정할 수 있는 subsample 매개변수도 지원함

##**[스태킹]**
- 사이킷런은 직접 지원하지 않음

#**[코드 필사]**

In [4]:
import warnings
warnings.filterwarnings('ignore')

# import package
import numpy as np
import os

# 5장에서의 moons dataset 불러오기
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
x,y = make_moons(n_samples=100, noise=0.15)
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

In [5]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

ada_clf=AdaBoostClassifier(
    DecisionTreeClassifier(max_depth=1), n_estimators=200,
    algorithm="SAMME", learning_rate=0.5)
ada_clf.fit(x_train, y_train)

AdaBoostClassifier(algorithm='SAMME',
                   estimator=DecisionTreeClassifier(max_depth=1),
                   learning_rate=0.5, n_estimators=200)

In [6]:
from sklearn.tree import DecisionTreeRegressor

tree_reg1=DecisionTreeRegressor(max_depth=2)
tree_reg1.fit(x,y)
y2=y-tree_reg1.predict(x)
tree_reg2=DecisionTreeRegressor(max_depth=2)
tree_reg2.fit(x,y2)

DecisionTreeRegressor(max_depth=2)

In [7]:
y3=y2-tree_reg2.predict(x)
tree_reg3=DecisionTreeRegressor(max_depth=2)
tree_reg3.fit(x,y3)

DecisionTreeRegressor(max_depth=2)

In [8]:
import numpy as np
x_new = np.array([[0.5, 1.2]])
y_pred=sum(tree.predict(x_new) for tree in (tree_reg1, tree_reg2, tree_reg3))

In [10]:
from sklearn.ensemble import GradientBoostingRegressor

gbrt=GradientBoostingRegressor(max_depth=2, n_estimators=3, learning_rate=1.0)
gbrt.fit(x,y)

GradientBoostingRegressor(learning_rate=1.0, max_depth=2, n_estimators=3)

In [11]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

x_train, x_val, y_train, y_val=train_test_split(x,y)
gbrt=GradientBoostingRegressor(max_depth=2, n_estimators=120)
gbrt.fit(x_train, y_train)

errors= [mean_squared_error(y_val, y_pred)
        for y_pred in gbrt.staged_predict(x_val)]
bst_n_estimators=np.argmin(errors)+1

gbrt_best=GradientBoostingRegressor(max_depth=2, n_estimators=bst_n_estimators)
gbrt_best.fit(x_train, y_train)

GradientBoostingRegressor(max_depth=2, n_estimators=np.int64(119))

In [12]:
gbrt=GradientBoostingRegressor(max_depth=2, warm_start=True)

min_val_error=float("inf")
error_going_up=0
for n_estimators in range(1,120):
    gbrt.n_estimators=n_estimators
    gbrt.fit(x_train, y_train)
    y_pred=gbrt.predict(x_val)
    val_error=mean_squared_error(y_val, y_pred)
    if val_error<min_val_error:
        min_val_error=val_error
        error_going_up=0
    else:
        error_going_up +=1
        if error_going_up==5:
            break

In [13]:
import xgboost

xgb_reg = xgboost.XGBRegressor(early_stopping_rounds=2)
xgb_reg.fit(x_train, y_train,
            eval_set = [(x_val, y_val)])
y_pred=xgb_reg.predict(x_val)

[0]	validation_0-rmse:0.40670
[1]	validation_0-rmse:0.35430
[2]	validation_0-rmse:0.33024
[3]	validation_0-rmse:0.32214
[4]	validation_0-rmse:0.32176
[5]	validation_0-rmse:0.32454


In [14]:
xgb_reg.fit(x_train, y_train,
            eval_set=[(x_val, y_val)])
y_pred=xgb_reg.predict(x_val)

[0]	validation_0-rmse:0.40670
[1]	validation_0-rmse:0.35430
[2]	validation_0-rmse:0.33024
[3]	validation_0-rmse:0.32214
[4]	validation_0-rmse:0.32176
[5]	validation_0-rmse:0.32454
